## Cluster upgrade with `kubeadm`

You don't `kubectl` your way to a new version — you run `kubeadm upgrade` on the nodes. The CKA may ask you to *describe* the order and the skew rules.

### The version-skew rule

- **`kubectl`** — within ±1 minor of the API server.
- **`kube-apiserver`** — the **highest** version in the cluster; everything else matches or trails.
- **controller-manager / scheduler / CCM** — same minor as the API server, or one older.
- **`kubelet`** — up to **two** minors older (1.28+; was one).
- **`kube-proxy`** — same minor as the node's kubelet.

So a 1.29 API server serves 1.28 and 1.27 kubelets, but a 1.27 API server **can't** serve a 1.29 kubelet: **control plane first, kubelets after.**

### Upgrade order

1. **First control-plane node** — `apt install kubeadm=<ver>` → `kubeadm upgrade plan` → `kubeadm upgrade apply v<ver>` → `drain` → upgrade `kubelet`+`kubectl` → restart kubelet → `uncordon`.
2. **Other control-plane nodes** — same, but `kubeadm upgrade node` (the first node did the cluster-wide work).
3. **Each worker** — drain, upgrade `kubeadm`+`kubelet`+`kubectl`, restart, uncordon.

### What `kubeadm upgrade` does

Pulls new control-plane images, **rotates certs** (a free renewal), updates the static-pod manifests with new tags (the kubelet restarts them), and updates kube-proxy + CoreDNS. It does **not** upgrade the kubelet package — you do that separately.

**One minor at a time** — 1.27 → 1.28 → 1.29, never skipping. Patches are free. This closes the architecture module: you can now name every component, install a cluster, secure it with TLS, back up etcd, and upgrade — the whole **Control Plane** box, operable end to end.